# Daily Challenge – Week 5, Day 1: Logistic Regression for Admission Prediction
Predict whether a student is admitted to university from their scores on two exams.
Dataset: `ex2data1.txt` (100 students: exam 1 score, exam 2 score, admitted 1/0).

## 1. Data Exploration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Works locally (file in this folder) and in Colab (reads the copy on GitHub)
LOCAL_FILE = "ex2data1.txt"
GITHUB_URL = "https://raw.githubusercontent.com/jenny-palti/DI-Bootcamp2026/main/Week5/Day1/ex2data1.txt"

# The file has no header row, so we name the columns ourselves
df = pd.read_csv(LOCAL_FILE if os.path.exists(LOCAL_FILE) else GITHUB_URL,
                 header=None, names=["exam1", "exam2", "admitted"])
print(df.shape)
df.head()

In [ ]:
print(df.describe().round(2))
print("\nAdmitted vs. not admitted:")
df["admitted"].value_counts()

In [ ]:
admitted = df[df["admitted"] == 1]
rejected = df[df["admitted"] == 0]

plt.figure(figsize=(7, 6))
plt.scatter(admitted["exam1"], admitted["exam2"], marker="+", s=70, c="green", label="Admitted")
plt.scatter(rejected["exam1"], rejected["exam2"], marker="o", s=40, c="red", label="Not admitted")
plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.title("Exam scores of admitted vs. not admitted students")
plt.legend()
plt.show()

The two groups are fairly well separated by a **diagonal line**: students with high scores on both exams are admitted. That's a good sign that logistic regression, which draws a straight decision boundary, will work well.

## 2. Applying Logistic Regression with scikit-learn

In [ ]:
X = df[["exam1", "exam2"]]
y = df["admitted"]

model = LogisticRegression()
model.fit(X, y)

print("Intercept (θ0):", round(model.intercept_[0], 3))
print("Coefficients (θ1, θ2):", model.coef_[0].round(3))

## 3. Making Predictions

In [ ]:
y_pred = model.predict(X)
accuracy = accuracy_score(y, y_pred)
print(f"Training accuracy: {accuracy:.0%}")
print("\nConfusion matrix (rows = true, columns = predicted):")
print(confusion_matrix(y, y_pred))

# Example: probability of admission for a student with 45 on exam 1 and 85 on exam 2
example = pd.DataFrame({"exam1": [45], "exam2": [85]})
print(f"\nStudent with scores 45 and 85 → probability of admission: {model.predict_proba(example)[0, 1]:.2f}")

In [ ]:
# Decision boundary: the line where the predicted probability is exactly 0.5
# θ0 + θ1·x1 + θ2·x2 = 0  →  x2 = −(θ0 + θ1·x1) / θ2
theta0 = model.intercept_[0]
theta1, theta2 = model.coef_[0]
x1_line = np.linspace(df["exam1"].min(), df["exam1"].max(), 100)
x2_line = -(theta0 + theta1 * x1_line) / theta2

plt.figure(figsize=(7, 6))
plt.scatter(admitted["exam1"], admitted["exam2"], marker="+", s=70, c="green", label="Admitted")
plt.scatter(rejected["exam1"], rejected["exam2"], marker="o", s=40, c="red", label="Not admitted")
plt.plot(x1_line, x2_line, "b-", label="Decision boundary (p = 0.5)")
plt.xlabel("Exam 1 score")
plt.ylabel("Exam 2 score")
plt.title(f"Logistic regression decision boundary – accuracy {accuracy:.0%}")
plt.legend()
plt.show()

## 4. Model Evaluation and Interpretation
- **The model:** logistic regression learned the equation **−25.05 + 0.205 × exam1 + 0.201 × exam2**, and passed it through the sigmoid function to get a probability of admission. Both coefficients are positive and almost equal, so **both exams count about the same**, and a higher score on either one raises the chance of admission.
- **Decision boundary:** a student is predicted "admitted" when that equation is above 0, which is roughly when **exam1 + exam2 > ~124**. On the plot this is the straight blue line separating the two groups.
- **Accuracy: 89%** (89 of 100 students classified correctly). The confusion matrix shows 6 rejected students predicted as admitted and 5 admitted students predicted as rejected; these are the points close to the boundary line, where the scores alone don't decide clearly.
- **Example:** a student with 45 on exam 1 and 85 on exam 2 has a **78%** predicted chance of admission.
- **Caveats:** the accuracy is measured on the **same data the model was trained on**, so it's an optimistic estimate. With only 100 students, a train/test split or cross-validation would give a more honest number. Real admissions also depend on more than two exam scores, so the remaining errors make sense.